# Wykład 06 - przykłady

Przykłady wymagają NumPy, SciPy i Matplotlib. W razie potrzeby zainstaluj je poleceniem `%pip install numpy scipy matplotlib` w osobnej komórce. 

## Broadcasting - odejmowanie tła - slajdy 11 i 12

Wiersze tablicy to serie pomiarowe, a kolumny to kanały detektora.
Wektor o kształcie `(3,)` zawiera po jednej wartości tła dla każdego kanału.
Przy odejmowaniu NumPy dopasowuje tę oś do ostatniej osi tablicy pomiarów.

Aby odjąć po jednej wartości od każdego wiersza, użyj `[:, None]` i zmień kształt wektora z `(2,)` na `(2, 1)`.
Bez dodatkowej osi NumPy porównuje rozmiary `3` i `2`, co powoduje `ValueError`.

In [ ]:
import numpy as np

pomiary = np.array([[10, 20, 30],
                   [40, 50, 60]])
tlo_kanalow = np.array([1, 2, 3])
tlo_serii = np.array([1, 4])

print("Pomiary:", pomiary.shape)
print("Tło kanałów:", tlo_kanalow.shape)
print(pomiary - tlo_kanalow)

try:
    pomiary - tlo_serii
except ValueError as blad:
    print("ValueError:", blad)

print("Tło serii po dodaniu osi:", tlo_serii[:, None].shape)
print(pomiary - tlo_serii[:, None])

## Broadcasting - normalizacja wierszy - slajd 19

Wyznaczamy średnią każdej serii wzdłuż osi kolumn (`axis=1`). Dzięki `keepdims=True` tablica średnich ma kształt `(2, 1)`, więc można podzielić każdy wiersz pomiarów przez jego średnią. W tym przykładzie obie średnie są dodatnie.

In [ ]:
import numpy as np

pomiary = np.array([[10, 20, 30],
                   [40, 50, 60]])

print("Średnie bez keepdims:", pomiary.mean(axis=1).shape)
srednie = pomiary.mean(axis=1, keepdims=True)
print("Średnie z keepdims:", srednie.shape)
print(srednie)

znormalizowane = pomiary / srednie
print("Znormalizowane pomiary:")
print(znormalizowane)
print("Średnia każdego wiersza:", znormalizowane.mean(axis=1))

## Transpozycja - slajd 21

In [ ]:
import numpy as np

A = np.array([[1, 2],
              [3, 4],
              [5, 6]])
A_T = A.T

print("Macierz A:", A.shape)
print(A)
print("\nTranspozycja macierzy A:", A_T.shape)
print(A_T)

## Wartości i wektory własne - slajd 23

Funkcja `np.linalg.eig` zwraca wartości własne oraz macierz, której kolumny są odpowiadającymi im wektorami własnymi. Sprawdzamy równanie $Av = \lambda v$ dla pierwszej pary, porównując wyniki z tolerancją na błędy zaokrągleń przez `np.allclose`.

In [ ]:
import numpy as np

A = np.array([[2, 1], [1, 2]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print("Wartości własne:", eigenvalues)
print("Wektory własne zapisane w kolumnach:")
print(eigenvectors)

i = 0
v = eigenvectors[:, i]
Av = A @ v
lambda_v = eigenvalues[i] * v

print("Av:", Av)
print("λv:", lambda_v)
print("Zgodność z tolerancją:", np.allclose(Av, lambda_v))

## Równanie oscylatora harmonicznego - slajd 34

Równanie $x'' + \omega^2 x = 0$ zapisujemy jako układ $x' = v$ oraz $v' = -\omega^2 x$. Funkcja `solve_ivp` oblicza obie składowe, a `sol.y[0]` zawiera położenie w chwilach zapisanych w `sol.t`.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

def harmonic_oscillator(t, y, omega):
    x, v = y
    return [v, -omega**2 * x]

omega = 2.0
y0 = [1.0, 0.0]
t_span = (0, 10)
t_eval = np.linspace(*t_span, 200)

sol = solve_ivp(harmonic_oscillator, t_span, y0,
                args=(omega,), t_eval=t_eval)

plt.figure()
plt.plot(sol.t, sol.y[0])
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Oscylator harmoniczny")
plt.show()

## Dopasowywanie krzywej - slajd 36

Generujemy dane z prostej $y = 3x + 2$ i dodajemy szum o rozkładzie normalnym. Funkcja `curve_fit` wyznacza współczynniki modelu minimalizujące sumę kwadratów odchyleń od danych.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

rng = np.random.default_rng(6)
x_data = np.linspace(0, 10, 100)
y_data = 3.0 * x_data + 2.0 + rng.normal(size=x_data.size)

def linear_model(x, a, b):
    return a * x + b

params, covariance = curve_fit(linear_model, x_data, y_data)
print("Parametry dopasowania:")
print("a =", params[0])
print("b =", params[1])

plt.figure()
plt.scatter(x_data, y_data, label="Dane")
plt.plot(x_data, linear_model(x_data, *params), "r",
         label="Dopasowanie")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Dopasowanie prostej")
plt.show()

## Transformacja Fouriera - slajd 40

Sygnał zawiera dwie sinusoidy o częstotliwościach 5 Hz i 20 Hz. Przy częstotliwości próbkowania 500 Hz `fftfreq` przypisuje częstotliwość każdemu elementowi wyniku FFT. Na wykresie pokazujemy moduł współczynników FFT w zakresie od 0 do 50 Hz.

In [ ]:
from scipy.fft import fft, fftfreq
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 1, 500, endpoint=False)
signal_complex = (np.sin(2 * np.pi * 5 * t)
                  + 0.5 * np.sin(2 * np.pi * 20 * t))

yf = fft(signal_complex)
xf = fftfreq(500, d=1 / 500)

plt.figure()
plt.plot(xf, np.abs(yf))
plt.xlim(0, 50)
plt.xlabel("Częstotliwość [Hz]")
plt.ylabel("|FFT|")
plt.title("Widmo sygnału")
plt.show()

## Interpolacja jednowymiarowa - slajd 41

Z wartości funkcji sinus w dziesięciu punktach wyznaczamy wartości pośrednie. `np.interp` łączy kolejne punkty odcinkami prostych, a `CubicSpline` tworzy funkcję sklejaną z wielomianów trzeciego stopnia. Obie krzywe przechodzą przez punkty wejściowe.

In [ ]:
from scipy.interpolate import CubicSpline
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 10, 10)
y = np.sin(x)
f_cubic = CubicSpline(x, y)
x_new = np.linspace(0, 10, 100)

plt.figure()
plt.plot(x, y, "o", label="Dane")
plt.plot(x_new, np.interp(x_new, x, y), "-",
         label="Interpolacja liniowa")
plt.plot(x_new, f_cubic(x_new), "--",
         label="Interpolacja sześcienna")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Interpolacja jednowymiarowa")
plt.show()